<a href="https://colab.research.google.com/github/MSagri05/Assignment-03-Unsupervised-Learning-Classical-NLP/blob/main/Assignment_3_Unsupervised_Learning_%26_Classical_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3: Unsupervised Learning
## Vancouver Business Licences Explorer

**Course:** IAT 461

**Student:** Manmeet Sagri (301545311)

## 1. Import Libraries

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

plt.style.use("default")

## 2. Load the Dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Convert the GeoJSON File into a DataFrame

In [4]:
file_path = "/content/drive/MyDrive/business-licences.geojson"

with open(file_path, "r") as f:
    geojson = json.load(f)

print(type(geojson))
print(geojson.keys())

<class 'dict'>
dict_keys(['type', 'features'])


In [5]:
rows = []

for feature in geojson["features"]:
    properties = feature["properties"]

    geometry = feature.get("geometry")

    if geometry and geometry["type"] == "Point":
        lon, lat = geometry["coordinates"]
    else:
        lon, lat = None, None

    rows.append({
        **properties,
        "longitude": lon,
        "latitude": lat
    })

df = pd.DataFrame(rows)

## 4. Initial Data Exploration

In [7]:
df.head()

,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geo_point_2d,longitude,latitude,geom
0,26,4944875,26-202017,00,1554429 B.C. LTD.,ULTRAMARINOS PEPE,Issued,2026-05-25T13:27:38-06:00,2026-12-31,Limited Service Food Establishment,...,None,V5Z 1S6,South Cambie,2.0,559.0,2026-07-25T01:09:02-06:00,"{'lon': -123.119718378987, 'lat': 49.256859194...",-123.119718,49.256859,NaN
1,26,4944879,26-202021,00,TELUS HEALTH CARE CENTRES INC. / CLINIQUES TEL...,X-Ray 505,Issued,2026-03-23T16:56:24-06:00,2026-12-31,Laboratory Services,...,None,V5Z 1H4,Fairview,48.0,308.0,2026-07-25T01:09:02-06:00,"{'lon': -123.121054317485, 'lat': 49.263014530...",-123.121054,49.263015,NaN
2,26,4944884,26-202026,00,ROSLYN PLACE APTS. LTD.,None,Issued,2026-02-27T15:28:19-07:00,2026-12-31,Long-term Rental,...,None,V5N 5R2,Grandview-Woodland,0.0,5369.0,2026-07-25T01:09:02-06:00,"{'lon': -123.071106732275, 'lat': 49.265351363...",-123.071107,49.265351,NaN
3,26,4944901,26-202043,00,CAN-ACHIEVE EDUCATION&CULTURE INC.,Allway Solutions,Issued,2026-02-23T17:18:41-07:00,2026-12-31,Business Support Services,...,None,V5X 0C3,Marpole,6.0,254.0,2026-07-25T01:09:02-06:00,"{'lon': -123.115976004173, 'lat': 49.210010909...",-123.115976,49.210011,NaN
4,26,4944903,26-202045,00,THE LONG TOQUE CLUB,None,Cancelled,None,None,Association or Society,...,CA,None,Mount Pleasant,1.0,NaN,2026-07-25T01:09:02-06:00,None,NaN,NaN,NaN


### 4.1 Dataset Information

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204259 entries, 0 to 204258
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   folderyear             204259 non-null  object 
 1   licencersn             204259 non-null  object 
 2   licencenumber          204259 non-null  object 
 3   licencerevisionnumber  204259 non-null  object 
 4   businessname           190777 non-null  object 
 5   businesstradename      77378 non-null   object 
 6   status                 204259 non-null  object 
 7   issueddate             175491 non-null  object 
 8   expireddate            175515 non-null  object 
 9   businesstype           204259 non-null  object 
 10  businesssubtype        21543 non-null   object 
 11  unit                   48949 non-null   object 
 12  unittype               48684 non-null   object 
 13  house                  109778 non-null  object 
 14  street                 109795 non-nu

### 4.2 Dataset Size

In [8]:
df.shape

(204259, 27)

### 4.3 Available Features

In [9]:
df.columns

Index(['folderyear', 'licencersn', 'licencenumber', 'licencerevisionnumber',
       'businessname', 'businesstradename', 'status', 'issueddate',
       'expireddate', 'businesstype', 'businesssubtype', 'unit', 'unittype',
       'house', 'street', 'city', 'province', 'country', 'postalcode',
       'localarea', 'numberofemployees', 'feepaid', 'extractdate',
       'geo_point_2d', 'longitude', 'latitude', 'geom'],
      dtype='object')

### 4.4 Missing Values


In [10]:
df.isnull().sum().sort_values(ascending=False)

,0
geom,204259
businesssubtype,182716
unittype,155575
unit,155310
businesstradename,126881
latitude,101627
longitude,101627
geo_point_2d,101627
postalcode,95219
house,94481


In [11]:
(df.isnull().mean() * 100).sort_values(ascending=False)

,0
geom,100.000000
businesssubtype,89.453096
unittype,76.165555
unit,76.035817
businesstradename,62.117704
latitude,49.753989
longitude,49.753989
geo_point_2d,49.753989
postalcode,46.616795
house,46.255489


### 4.5 Business Licence Status

In [13]:
df["status"].value_counts()

,count
status,
Issued,167167
Pending,14793
Gone Out of Business,10520
Inactive,6590
Cancelled,5189


### 4.6 Business Type Distribution

In [12]:
df["businesstype"].value_counts()

,count
businesstype,
Long-term Rental,45432
Health Care Professionals and Services,18759
General Contractor,15907
Short-term Rental Operator,13418
Retail Dealer,9455
...,...
Urban Farm Class B,8
Marine Service Station,6
Adult Services,4


### 4.7 Business Subtype Distribution

In [14]:
df["businesssubtype"].value_counts()

,count
businesssubtype,
Class 1 with liquor service,4149
Without Liquor,3036
Class 1 no liquor service,1975
Beauty and Wellness Centre,1719
Barber Shop or Beauty Salon,1621
...,...
Concert: 1000-2000 people,1
Motor Vessel,1
Transient Trader / Peddler - Annual,1


### 4.8 Number of Employees

In [15]:
df["numberofemployees"].describe()

,numberofemployees
count,204259.000000
mean,10.021531
std,72.771582
min,0.000000
25%,0.000000
50%,1.000000
75%,4.000000
max,5876.000000


### 4.9 Licence Fees

In [16]:
df["feepaid"].describe()

,feepaid
count,128840.000000
mean,516.801428
std,1109.697928
min,2.000000
25%,207.000000
50%,277.000000
75%,405.000000
max,63722.000000


### 4.10 initial observations

the dataset contains 204,259 business licence records and 27 columns. it includes information about licence status, business type, business subtype, number of employees, fees paid, dates, addresses, neighbourhoods, postal codes, and geographic coordinates.

from the initial exploration, the dataset has several missing values that will need to be handled before clustering. the 'geom' column is completely empty, while 'businesssubtype', 'unit', 'unittype', and 'businesstradename' are also missing for a large number of records. around half of the records are missing latitude and longitude, which means those rows cannot be used for the location-based clustering in part a2. the 'feepaid' column is missing for around 37% of the records, while 'numberofemployees' has no missing values but contains a very large maximum value compared to the median, which may indicate outliers.

the dataset also includes multiple licence statuses, although most records are listed as issued. there are 94 business types, with a few very common categories and many categories with only a small number of records. based on these observations, the next step will be to filter the dataset, handle missing values, remove unusable geographic records, review possible outliers, and consolidate sparse business type categories before applying clustering.

## 5. data cleaning

before applying clustering algorithms, the dataset needs to be cleaned and prepared. in this section, i identify and justify the cleaning decisions that will be used throughout the rest of the assignment.

### 5.1 Remove Empty Columns

In [17]:
df = df.drop(columns=["geom"])

the 'geom' column contains no values for any of the records, so it does not provide any useful information. since the column is completely empty, it was removed from the dataset.

In [22]:
print(df.shape)

(85938, 26)


### 5.2 Keep Only Active Licences

In [18]:
df = df[df["status"] == "Issued"]

In [19]:
df["status"].value_counts()

,count
status,
Issued,167167


only businesses with an 'issued' licence were kept for the analysis. this removes businesses that are pending, cancelled, inactive, or no longer operating. using active licences provides a more consistent dataset for clustering because it represents businesses that are currently licensed.

### 5.3 Remove Records Without Geographic Coordinates

In [20]:
df = df.dropna(subset=["latitude","longitude"])

In [21]:
df["feepaid"].isnull().sum()

np.int64(26482)